# Demo — End-to-end music context inference

Load a sample clip, build/load its segment graph, encode caption text with DistilBERT, and run Task 3 fusion (or Task 1 tags / Task 4 retrieval) using saved checkpoints.

In [ ]:
from pathlib import Path
import sys
import torch
import numpy as np

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.utils import load_config, load_json, get_device
from src.bert_encoder import encode_texts, get_tokenizer
from src.fusion_model import FusionModel
from src.graph_builder import build_graphs_from_audio

cfg = load_config(ROOT / 'config.yaml')
device = get_device(cfg)
splits = load_json(Path(cfg['paths']['splits']) / 'musiccaps.json')
vocab = splits['vocab']
sample = splits['val'][0] if splits['val'] else splits['train'][0]
print('Sample:', sample['ytid'])
print('Caption:', sample.get('text', sample.get('caption', ''))[:300])

In [ ]:
audio = Path(sample['audio_path'])
gpath = Path(cfg['paths']['processed']) / 'musiccaps' / f"{sample['ytid']}_segment.pt"
if gpath.exists():
    g = torch.load(gpath, map_location='cpu', weights_only=False)
    x, ei = g['x'].float(), g['edge_index'].long()
else:
    graphs = build_graphs_from_audio(audio, cfg)
    from src.graph_builder import graph_to_torch
    gt = graph_to_torch(graphs['segment'])
    x, ei = gt['x'], gt['edge_index']
print('nodes', x.shape, 'edges', ei.shape)

In [ ]:
ckpt_path = Path(cfg['paths']['checkpoints']) / 'task3_fusion_cross_attention.pt'
tokenizer = get_tokenizer(cfg['model']['bert_name'])
model = FusionModel(
    in_node_dim=x.shape[-1],
    num_labels=len(vocab),
    bert_name=cfg['model']['bert_name'],
    gnn_hidden=cfg['model']['gnn_hidden'],
    mode='cross_attention',
).to(device)
if ckpt_path.exists():
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(state['model'], strict=False)
    print('Loaded', ckpt_path)
else:
    print('Checkpoint missing — using randomly initialized weights for demo wiring only.')
model.eval()
enc = encode_texts([sample.get('text') or sample.get('caption')], tokenizer, cfg['data']['max_text_len'], device)
with torch.no_grad():
    out = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'], x=x.to(device), edge_index=ei.to(device))
    prob = torch.sigmoid(out['logits'])[0].cpu().numpy()
top = np.argsort(-prob)[:10]
print('Top predicted aspects:')
for j in top:
    print(f'  {vocab[j]:20s} {prob[j]:.3f}')

## Optional: Task 4 retrieval
See `results/retrieval_examples/` for qualitative caption→audio matches after `python -m src.train --task 4`.